# Gold Layer — Dimension: Date
## SalesFlow Data Lakehouse | Phase 5: Analytical Layer

Generates a complete date dimension programmatically — no source table needed.
Covers the period from **2020-01-01** to **2030-12-31** (4,018 dates).

**Design:**
| Column | Type | Description |
|---|---|---|
| `date_key` | PK | Integer in format `YYYYMMDD` (e.g. `20260201`) |
| `full_date` | date | Full date value |
| `year` | int | Calendar year (e.g. `2026`) |
| `quarter` | int | Quarter of the year (1–4) |
| `month` | int | Month number (1–12) |
| `month_name` | string | Month name (e.g. `January`) |
| `day` | int | Day of the month (1–31) |
| `day_of_week` | int | Day of week (1=Sunday, 7=Saturday) |
| `day_name` | string | Day name (e.g. `Monday`) |
| `week_of_year` | int | ISO week number (1–53) |
| `is_weekend` | boolean | TRUE if Saturday or Sunday |

**Period:** 2020-01-01 → 2030-12-31


## 1. Generate Date Range Programmatically
Uses `datetime` and `timedelta` to iterate over every date in the period
and build a list of rows, which is then converted to a Spark DataFrame.

In [0]:
from datetime import date, timedelta

# Define period boundaries
start_date = date(2020, 1, 1)
end_date   = date(2030, 12, 31)

# Day name and month name lookups
day_names = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday",
    3: "Thursday", 4: "Friday", 5: "Saturday", 6: "Sunday"
}
month_names = {
    1: "January", 2: "February", 3: "March", 4: "April",
    5: "May", 6: "June", 7: "July", 8: "August",
    9: "September", 10: "October", 11: "November", 12: "December"
}

# Generate one row per date
rows = []
current = start_date

while current <= end_date:
    # day_of_week: Python weekday() returns 0=Monday, 6=Sunday
    # We remap to 1=Sunday, 7=Saturday as per the spec
    python_weekday = current.weekday()  # 0=Mon, 6=Sun
    day_of_week = (python_weekday + 2) % 7  # remap: Sun=1, Sat=7
    if day_of_week == 0:
        day_of_week = 7  # Saturday

    rows.append((
        int(current.strftime("%Y%m%d")),    # date_key  e.g. 20260201
        current,                             # full_date
        current.year,                        # year
        (current.month - 1) // 3 + 1,       # quarter
        current.month,                       # month
        month_names[current.month],          # month_name
        current.day,                         # day
        day_of_week,                         # day_of_week (1=Sun, 7=Sat)
        day_names[python_weekday],           # day_name
        int(current.strftime("%W")) + 1,     # week_of_year (1-based)
        python_weekday >= 5                  # is_weekend (Sat=5, Sun=6)
    ))

    current += timedelta(days=1)

print(f"Total dates generated: {len(rows)}")

## 2. Create Spark DataFrame
Convert the Python list to a Spark DataFrame with explicit schema.

In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DateType, StringType, BooleanType
)

# Define explicit schema — never rely on inferSchema for generated data
schema = StructType([
    StructField("date_key",     IntegerType(), False),  # NOT NULL — PK
    StructField("full_date",    DateType(),    False),
    StructField("year",         IntegerType(), False),
    StructField("quarter",      IntegerType(), False),
    StructField("month",        IntegerType(), False),
    StructField("month_name",   StringType(),  False),
    StructField("day",          IntegerType(), False),
    StructField("day_of_week",  IntegerType(), False),
    StructField("day_name",     StringType(),  False),
    StructField("week_of_year", IntegerType(), False),
    StructField("is_weekend",   BooleanType(), False),
])

df = spark.createDataFrame(rows, schema=schema)

print(f"DataFrame created with {df.count()} rows and {len(df.columns)} columns")
display(df.limit(10))

## 3. Save as Delta Table

In [0]:
# Write to Gold layer as Delta table — overwrite for first load
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("salesflow_dev.gold.dim_date")

print("Table saved: salesflow_dev.gold.dim_date")

## 4. Validation

In [0]:
from pyspark.sql.functions import col

dim_date = spark.table("salesflow_dev.gold.dim_date")

# Record count — expect 4,018 dates (2020 to 2030 inclusive)
print(f"Total records: {dim_date.count()}")

# Date range — confirm boundaries
print("\nDate range:")
display(dim_date.select("full_date").summary("min", "max"))

# date_key uniqueness — must be 0 duplicates
duplicate_keys = dim_date.groupBy("date_key").count().filter(col("count") > 1)
print(f"\nDuplicate date_keys (expected 0): {duplicate_keys.count()}")

# Verify day_of_week distribution — should be roughly equal across 7 days
print("\nRecords by day_of_week:")
display(dim_date.groupBy("day_of_week", "day_name").count().orderBy("day_of_week"))

# Verify weekend flag
print("\nWeekend distribution:")
display(dim_date.groupBy("is_weekend").count())

# Spot check — a known date to validate all columns
print("\nSpot check — 2026-02-01 (a Sunday):")
display(dim_date.filter(col("date_key") == 20260201))

# Schema
print("\nSchema:")
dim_date.printSchema()